# Dataset 5 — Wind & Solar Energy Production (Kaggle)
## Etapa B — Python / Pandas

Pré-requisito: Etapa A no Orange (Select Columns mantendo atributos temporais + variáveis de produção solar e eólica; checar min/max e ausentes; amostra aleatória de 20%; exportar CSV).

**Atenção:** os nomes das colunas de produção solar e eólica variam conforme a versão exportada do Kaggle/Orange. Ajuste `COLUNA_SOLAR_ORIGINAL` e `COLUNA_EOLICA_ORIGINAL` abaixo para os nomes reais presentes no seu CSV (verifique com `df.columns` na primeira célula).

In [19]:
import pandas as pd

CAMINHO_CSV = "Wind & Solar Energy Production.csv"

df = pd.read_csv(CAMINHO_CSV)
df.columns

Index(['Date', 'Day_of_Year', 'Day_Name', 'Month_Name', 'Season', 'Production',
       'Source'],
      dtype='object')

### 0. Checagem de ausentes (equivalente ao passo 4 da Etapa A, replicado aqui para conferência)

In [20]:
df.isnull().sum()

,0
Date,0
Day_of_Year,0
Day_Name,0
Month_Name,0
Season,0
Production,0
Source,0


### 1. Renomear variáveis principais

In [25]:
df["Geracao_Solar"] = df["Production"].where(df["Source"] == "Solar")
df["Geracao_Eolica"] = df["Production"].where(df["Source"] == "Wind")

df.head()

,Date,Day_of_Year,Day_Name,Month_Name,Season,Production,Source,Geracao_Solar,Geracao_Eolica
0,12/18/2022,352,Sunday,December,Winter,11400,Wind,NaN,11400.0
1,1/29/2024,29,Monday,January,Winter,7917,Wind,NaN,7917.0
2,8/1/2024,214,Thursday,August,Summer,8835,Solar,8835.0,NaN
3,11/10/2020,315,Tuesday,November,Fall,989,Wind,NaN,989.0
4,12/19/2022,353,Monday,December,Winter,12526,Wind,NaN,12526.0


### 2. Valor máximo de cada fonte

In [26]:
max_solar = df["Geracao_Solar"].max()
max_eolica = df["Geracao_Eolica"].max()

print(f"Máximo Geração Solar: {max_solar}")
print(f"Máximo Geração Eólica: {max_eolica}")

Máximo Geração Solar: 16316.0
Máximo Geração Eólica: 22929.0


### 3. Limiar de 70% do máximo de cada fonte, separadamente

In [27]:
limiar_solar = 0.70 * max_solar
limiar_eolica = 0.70 * max_eolica

print(f"Limiar (70% do máximo) - Solar: {limiar_solar}")
print(f"Limiar (70% do máximo) - Eólica: {limiar_eolica}")

Limiar (70% do máximo) - Solar: 11421.199999999999
Limiar (70% do máximo) - Eólica: 16050.3


### 4 e 5. DataFrames de alta geração por fonte, contagem e percentual

In [28]:
df_alta_solar = df[df["Geracao_Solar"] > limiar_solar]
df_alta_eolica = df[df["Geracao_Eolica"] > limiar_eolica]

qtd_alta_solar = len(df_alta_solar)
qtd_alta_eolica = len(df_alta_eolica)

percentual_alta_solar = qtd_alta_solar / len(df) * 100
percentual_alta_eolica = qtd_alta_eolica / len(df) * 100

print(f"Alta geração solar: {qtd_alta_solar} registros ({percentual_alta_solar:.2f}% da amostra)")
print(f"Alta geração eólica: {qtd_alta_eolica} registros ({percentual_alta_eolica:.2f}% da amostra)")

Alta geração solar: 45 registros (0.43% da amostra)
Alta geração eólica: 245 registros (2.36% da amostra)


### 6. Comparação de frequência entre as fontes

In [31]:
if percentual_alta_solar > percentual_alta_eolica:
    fonte_mais_frequente = "solar"
elif percentual_alta_eolica > percentual_alta_solar:
    fonte_mais_frequente = "eólica"
else:
    fonte_mais_frequente = "empate"

print(f"Fonte com maior frequência de registros acima de 70% do próprio máximo: {fonte_mais_frequente}")

Fonte com maior frequência de registros acima de 70% do próprio máximo: eólica


In [30]:
from IPython.display import display, Markdown

display(Markdown(f"""
**Interpretação:**

A fonte **{fonte_mais_frequente}** apresentou maior percentual de registros acima de 70% do seu próprio máximo ({percentual_alta_solar:.2f}% solar vs. {percentual_alta_eolica:.2f}% eólica).
"""))



**Interpretação:**

A fonte **eólica** apresentou maior percentual de registros acima de 70% do seu próprio máximo (0.43% solar vs. 2.36% eólica).


### 7. Por que não usar o mesmo valor numérico de limiar para as duas fontes

Solar e eólica têm escalas de geração diferentes — os valores de máximo (`max_solar` e `max_eolica`) normalmente não são iguais, e a distribuição de cada variável ao longo do tempo também difere (a solar tem padrão diário/sazonal ligado à irradiância, a eólica depende de regime de vento, sem ciclo diário fixo). Usar um único valor absoluto de potência como limiar comum implicaria:

- Favorecer artificialmente a fonte com maior amplitude bruta: se, por exemplo, `max_eolica` for numericamente maior que `max_solar`, um limiar fixo baseado no valor eólico tornaria praticamente impossível a solar ultrapassá-lo, mesmo em seus picos reais.
- Perder o significado relativo do "alto desempenho": 70% do próprio máximo representa a mesma posição relativa dentro da capacidade de cada fonte, permitindo comparar a *frequência de picos relativos* de cada uma em pé de igualdade — o que um limiar absoluto comum não garante.
- Ignorar diferenças de unidade/escala de captura dos dados (dependendo do dataset, geração solar e eólica podem vir em unidades ou faixas de instrumentação distintas), o que tornaria a comparação numérica direta sem sentido físico.

Por isso, calcular o limiar separadamente como 70% do máximo de cada variável é o que torna a comparação entre solar e eólica válida.